# Seeded Run 2

This notebook trains five matched seeded pairs for the 2-class SHD odd/even parity task.

Each training cell handles one seed pair independently. Once a pair finishes, its checkpoints and summaries are already saved, so you can stop there and later continue with the next seed cell instead of rerunning a full loop.

Cell 2 defines the seeds used in this notebook. Cell 3 verifies three things before training starts:
1. this notebook is locked to the 2-class odd/even classification task
2. the homogeneous and heterogeneous model definitions are set up correctly
3. the log-normal hidden tau_m samples vary across the configured seeds

In [8]:
from pathlib import Path
import importlib

import pandas as pd
from IPython.display import display

import seeded_runs_common as seeded_runs_common

seeded_runs_common = importlib.reload(seeded_runs_common)

CHECKPOINT_ROOT = seeded_runs_common.CHECKPOINT_ROOT
DEVICE = seeded_runs_common.DEVICE
SHD_TEST = seeded_runs_common.SHD_TEST
SHD_TRAIN = seeded_runs_common.SHD_TRAIN
TAU_ARTIFACT_PATH = seeded_runs_common.TAU_ARTIFACT_PATH
TASKS = seeded_runs_common.TASKS
build_pair_summary_row = seeded_runs_common.build_pair_summary_row
build_sampling_preview_rows = seeded_runs_common.build_sampling_preview_rows
build_seeded_pair = seeded_runs_common.build_seeded_pair
expected_2class_odd_even_map = seeded_runs_common.expected_2class_odd_even_map
load_default_caches = seeded_runs_common.load_default_caches
read_manifest_rows = seeded_runs_common.read_manifest_rows
run_pair_training = seeded_runs_common.run_pair_training
upsert_rows = seeded_runs_common.upsert_rows
write_manifest_rows = seeded_runs_common.write_manifest_rows

RUN_LABEL = "seeded_run_2"
TASK_KEY = "2class"
MEM_DISTRIBUTION_FAMILY = "lognormal"
MASTER_SEEDS = [201, 211, 221, 231, 241]

RUN_DIR = CHECKPOINT_ROOT / RUN_LABEL / f"{TASK_KEY}_{MEM_DISTRIBUTION_FAMILY}"
RESULT_STEM = RUN_DIR / f"{RUN_LABEL}_checkpoint_summary"
PAIR_STEM = RUN_DIR / f"{RUN_LABEL}_pair_summary"
PAIRED_ACC_CSV = RUN_DIR / f"{RUN_LABEL}_paired_accuracy.csv"

TASK_DEF = TASKS[TASK_KEY]
EXPECTED_LABEL_MAP = expected_2class_odd_even_map()

print(f"Device: {DEVICE}")
print(f"Train file exists: {SHD_TRAIN.exists()}")
print(f"Test file exists: {SHD_TEST.exists()}")
print(f"Tau artifact exists: {TAU_ARTIFACT_PATH.exists()}")
print(f"Run directory: {RUN_DIR}")
print(f"Master seeds: {MASTER_SEEDS}")
print(f"Task name: {TASK_DEF['task_name']}")
print(f"Task outputs: {TASK_DEF['nb_outputs']}")

Device: cuda
Train file exists: True
Test file exists: True
Tau artifact exists: True
Run directory: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\Checkpoints\SeededRuns\seeded_run_2\2class_lognormal
Master seeds: [201, 211, 221, 231, 241]
Task name: binary_parity
Task outputs: 2


In [9]:
assert TASK_KEY == "2class"
assert TASK_DEF["nb_outputs"] == 2
assert TASK_DEF["task_name"] == "binary_parity"
assert TASK_DEF["task_label_map"] == EXPECTED_LABEL_MAP

preview = build_seeded_pair(
    master_seed=MASTER_SEEDS[0],
    task_key=TASK_KEY,
    mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
)
preview_meta = preview["metadata"]

assert preview["hom_prms"]["nb_outputs"] == 2
assert preview["hetero_prms"]["nb_outputs"] == 2
assert not preview["hom_model"].network[0].alpha.requires_grad
assert not preview["hom_model"].network[0].beta.requires_grad
assert not preview["hetero_model"].network[0].alpha.requires_grad
assert not preview["hetero_model"].network[0].beta.requires_grad
assert preview_meta["linear_sync_verified"]
assert preview_meta["hom_hidden_tau_unique"] == 1
assert preview_meta["hetero_hidden_tau_unique"] > 1

sampling_rows = build_sampling_preview_rows(
    MASTER_SEEDS,
    task_key=TASK_KEY,
    mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
)
sampling_df = pd.DataFrame(sampling_rows).sort_values("pair_seed").reset_index(drop=True)

assert not sampling_df["sample_matches_previous"].any()

display(sampling_df)
print("2-class odd/even task mapping verified.")
print("Homogeneous anchor and heterogeneous sampled definitions verified.")
print("Log-normal hidden tau_m sampling varies across the configured seeds.")

,pair_seed,task_key,task_name,mem_distribution_family,linear_sync_verified,hetero_hidden_tau_unique,hom_hidden_tau_unique,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,sample_matches_previous
0,201,2class,binary_parity,lognormal,True,30,1,23.203168,3.287719,99.749887,34.831960,31.291707,False
1,211,2class,binary_parity,lognormal,True,32,1,28.072028,2.590169,99.749887,37.446883,26.115192,False
2,221,2class,binary_parity,lognormal,True,32,1,24.333673,4.329914,99.749887,32.422747,22.852126,False
3,231,2class,binary_parity,lognormal,True,29,1,25.226301,2.072141,99.749887,37.873874,31.872785,False
4,241,2class,binary_parity,lognormal,True,29,1,29.445742,3.794687,99.749887,41.174600,31.371058,False


2-class odd/even task mapping verified.
Homogeneous anchor and heterogeneous sampled definitions verified.
Log-normal hidden tau_m sampling varies across the configured seeds.


In [10]:
train_cache, test_cache = load_default_caches()

Pre-loading SHD training data into RAM...
  SHDCache: 8156 samples loaded from shd_train.h5
Pre-loading SHD test data into RAM...
  SHDCache: 2264 samples loaded from shd_test.h5


In [11]:
CHECKPOINT_KEY_FIELDS = ["pair_seed", "pair_role"]
PAIR_KEY_FIELDS = ["pair_seed"]

def show_run_status():
    checkpoint_rows = read_manifest_rows(RESULT_STEM)
    pair_rows = read_manifest_rows(PAIR_STEM)

    checkpoint_df = pd.DataFrame(checkpoint_rows)
    pair_df = pd.DataFrame(pair_rows)

    if not checkpoint_df.empty:
        checkpoint_df = checkpoint_df.sort_values(["pair_seed", "pair_role"]).reset_index(drop=True)
    if not pair_df.empty:
        pair_df = pair_df.sort_values("pair_seed").reset_index(drop=True)

    paired_acc_df = pd.DataFrame()
    if not checkpoint_df.empty:
        paired_acc_df = (
            checkpoint_df.pivot(index="pair_seed", columns="pair_role", values="final_test_acc")
            .reset_index()
            .sort_values("pair_seed")
            .reset_index(drop=True)
        )
        if {"heterogeneous_sampled", "homogeneous_anchor"}.issubset(paired_acc_df.columns):
            paired_acc_df["hetero_minus_hom"] = (
                paired_acc_df["heterogeneous_sampled"] - paired_acc_df["homogeneous_anchor"]
            )
            paired_acc_df.to_csv(PAIRED_ACC_CSV, index=False)

    return pair_df, checkpoint_df, paired_acc_df

def train_one_seed(master_seed):
    run_rows, pair_meta = run_pair_training(
        master_seed=master_seed,
        train_cache=train_cache,
        test_cache=test_cache,
        task_key=TASK_KEY,
        mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
        run_label=RUN_LABEL,
        skip_existing=True,
    )

    checkpoint_rows = upsert_rows(
        read_manifest_rows(RESULT_STEM),
        run_rows,
        CHECKPOINT_KEY_FIELDS,
    )
    pair_rows = upsert_rows(
        read_manifest_rows(PAIR_STEM),
        [build_pair_summary_row(pair_meta)],
        PAIR_KEY_FIELDS,
    )

    write_manifest_rows(checkpoint_rows, RESULT_STEM)
    write_manifest_rows(pair_rows, PAIR_STEM)

    pair_df, checkpoint_df, paired_acc_df = show_run_status()
    seed_df = checkpoint_df[checkpoint_df["pair_seed"] == master_seed].reset_index(drop=True)
    return seed_df, pair_df, checkpoint_df, paired_acc_df

print("Run helpers ready.")
print("Execute one seed cell at a time. Finished seeds are reused from saved checkpoints.")

Run helpers ready.
Execute one seed cell at a time. Finished seeds are reused from saved checkpoints.


In [12]:
# Train or reuse seed pair 201
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(MASTER_SEEDS[0])
display(seed_df)
display(pair_df[pair_df["pair_seed"] == MASTER_SEEDS[0]].reset_index(drop=True))


Seed 201 :: training homogeneous_anchor...
  epoch=001  train_acc=0.498  test_acc=0.505  (2.2 min)
  epoch=002  train_acc=0.497  test_acc=0.506  (2.1 min)
  epoch=003  train_acc=0.498  test_acc=0.505  (2.1 min)
  epoch=004  train_acc=0.498  test_acc=0.506  (2.1 min)
  epoch=005  train_acc=0.529  test_acc=0.701  (2.1 min)
  epoch=006  train_acc=0.713  test_acc=0.682  (2.1 min)
  epoch=007  train_acc=0.751  test_acc=0.708  (2.1 min)
  epoch=008  train_acc=0.780  test_acc=0.751  (2.4 min)
  epoch=009  train_acc=0.799  test_acc=0.785  (2.9 min)
  epoch=010  train_acc=0.809  test_acc=0.811  (2.8 min)
  epoch=011  train_acc=0.817  test_acc=0.748  (2.8 min)
  epoch=012  train_acc=0.831  test_acc=0.801  (2.8 min)
  epoch=013  train_acc=0.853  test_acc=0.800  (3.0 min)
  epoch=014  train_acc=0.859  test_acc=0.782  (3.1 min)
  epoch=015  train_acc=0.870  test_acc=0.813  (2.9 min)
  epoch=016  train_acc=0.878  test_acc=0.817  (2.9 min)
  epoch=017  train_acc=0.885  test_acc=0.834  (2.8 min)
  ep

,run_label,task_key,task_name,pair_seed,pair_role,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hidden_tau_mem_min_ms,hidden_tau_mem_max_ms,hidden_tau_mem_mean_ms,hidden_tau_mem_std_ms,linear_sync_verified,checkpoint,elapsed_s,final_test_acc,final_test_loss
0,seeded_run_2,2class,binary_parity,201,heterogeneous_sampled,lognormal,10.0,23.203168,3.287719,99.749887,34.831960,31.291707,True,C:\Users\Priya\Desktop\research project (SNN I...,3201.039513,0.864841,0.360071
1,seeded_run_2,2class,binary_parity,201,homogeneous_anchor,lognormal,10.0,23.203168,23.203168,23.203168,23.203168,0.000000,True,C:\Users\Priya\Desktop\research project (SNN I...,3918.951533,0.844965,0.404792


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,201,2class,binary_parity,lognormal,10.0,23.203168,3.287719,99.749887,34.83196,31.291707,30,1,True,6


In [13]:
# Train or reuse seed pair 211
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(MASTER_SEEDS[1])
display(seed_df)
display(pair_df[pair_df["pair_seed"] == MASTER_SEEDS[1]].reset_index(drop=True))


Seed 211 :: training homogeneous_anchor...
  epoch=001  train_acc=0.503  test_acc=0.508  (2.2 min)
  epoch=002  train_acc=0.504  test_acc=0.511  (2.2 min)
  epoch=003  train_acc=0.515  test_acc=0.521  (2.2 min)
  epoch=004  train_acc=0.586  test_acc=0.545  (2.2 min)
  epoch=005  train_acc=0.673  test_acc=0.710  (2.2 min)
  epoch=006  train_acc=0.721  test_acc=0.750  (2.2 min)
  epoch=007  train_acc=0.750  test_acc=0.764  (2.2 min)
  epoch=008  train_acc=0.759  test_acc=0.693  (2.2 min)
  epoch=009  train_acc=0.791  test_acc=0.769  (2.2 min)
  epoch=010  train_acc=0.807  test_acc=0.784  (2.2 min)
  epoch=011  train_acc=0.819  test_acc=0.662  (2.2 min)
  epoch=012  train_acc=0.823  test_acc=0.735  (2.2 min)
  epoch=013  train_acc=0.834  test_acc=0.760  (2.2 min)
  epoch=014  train_acc=0.841  test_acc=0.758  (2.2 min)
  epoch=015  train_acc=0.851  test_acc=0.737  (2.2 min)
  epoch=016  train_acc=0.854  test_acc=0.776  (2.2 min)
  epoch=017  train_acc=0.856  test_acc=0.712  (2.2 min)
  ep

,run_label,task_key,task_name,pair_seed,pair_role,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hidden_tau_mem_min_ms,hidden_tau_mem_max_ms,hidden_tau_mem_mean_ms,hidden_tau_mem_std_ms,linear_sync_verified,checkpoint,elapsed_s,final_test_acc,final_test_loss
0,seeded_run_2,2class,binary_parity,211,heterogeneous_sampled,lognormal,10.0,28.072028,2.590169,99.749887,37.446883,26.115192,True,C:\Users\Priya\Desktop\research project (SNN I...,3222.487701,0.506184,0.582101
1,seeded_run_2,2class,binary_parity,211,homogeneous_anchor,lognormal,10.0,28.072028,28.072028,28.072028,28.072028,0.000000,True,C:\Users\Priya\Desktop\research project (SNN I...,3238.934857,0.758834,0.504108


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,211,2class,binary_parity,lognormal,10.0,28.072028,2.590169,99.749887,37.446883,26.115192,32,1,True,6


In [14]:
# Train or reuse seed pair 221
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(MASTER_SEEDS[2])
display(seed_df)
display(pair_df[pair_df["pair_seed"] == MASTER_SEEDS[2]].reset_index(drop=True))


Seed 221 :: training homogeneous_anchor...
  epoch=001  train_acc=0.601  test_acc=0.644  (2.2 min)
  epoch=002  train_acc=0.667  test_acc=0.684  (2.2 min)
  epoch=003  train_acc=0.706  test_acc=0.676  (2.2 min)
  epoch=004  train_acc=0.720  test_acc=0.716  (2.1 min)
  epoch=005  train_acc=0.742  test_acc=0.716  (2.1 min)
  epoch=006  train_acc=0.763  test_acc=0.749  (2.1 min)
  epoch=007  train_acc=0.789  test_acc=0.739  (2.1 min)
  epoch=008  train_acc=0.804  test_acc=0.771  (2.1 min)
  epoch=009  train_acc=0.815  test_acc=0.770  (2.1 min)
  epoch=010  train_acc=0.832  test_acc=0.784  (2.1 min)
  epoch=011  train_acc=0.846  test_acc=0.744  (2.1 min)
  epoch=012  train_acc=0.857  test_acc=0.757  (2.1 min)
  epoch=013  train_acc=0.860  test_acc=0.761  (2.1 min)
  epoch=014  train_acc=0.863  test_acc=0.735  (2.1 min)
  epoch=015  train_acc=0.877  test_acc=0.767  (2.1 min)
  epoch=016  train_acc=0.880  test_acc=0.759  (2.1 min)
  epoch=017  train_acc=0.884  test_acc=0.818  (2.2 min)
  ep

,run_label,task_key,task_name,pair_seed,pair_role,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hidden_tau_mem_min_ms,hidden_tau_mem_max_ms,hidden_tau_mem_mean_ms,hidden_tau_mem_std_ms,linear_sync_verified,checkpoint,elapsed_s,final_test_acc,final_test_loss
0,seeded_run_2,2class,binary_parity,221,heterogeneous_sampled,lognormal,10.0,24.333673,4.329914,99.749887,32.422747,22.852126,True,C:\Users\Priya\Desktop\research project (SNN I...,3211.423915,0.822880,0.446224
1,seeded_run_2,2class,binary_parity,221,homogeneous_anchor,lognormal,10.0,24.333673,24.333673,24.333673,24.333673,0.000000,True,C:\Users\Priya\Desktop\research project (SNN I...,3204.421216,0.809187,0.502720


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,221,2class,binary_parity,lognormal,10.0,24.333673,4.329914,99.749887,32.422747,22.852126,32,1,True,6


In [15]:
# Train or reuse seed pair 231
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(MASTER_SEEDS[3])
display(seed_df)
display(pair_df[pair_df["pair_seed"] == MASTER_SEEDS[3]].reset_index(drop=True))


Seed 231 :: training homogeneous_anchor...
  epoch=001  train_acc=0.499  test_acc=0.505  (2.1 min)
  epoch=002  train_acc=0.499  test_acc=0.515  (2.1 min)
  epoch=003  train_acc=0.504  test_acc=0.510  (2.1 min)
  epoch=004  train_acc=0.577  test_acc=0.612  (2.2 min)
  epoch=005  train_acc=0.705  test_acc=0.578  (2.2 min)
  epoch=006  train_acc=0.753  test_acc=0.686  (2.2 min)
  epoch=007  train_acc=0.778  test_acc=0.770  (2.2 min)
  epoch=008  train_acc=0.801  test_acc=0.702  (2.2 min)
  epoch=009  train_acc=0.810  test_acc=0.742  (2.2 min)
  epoch=010  train_acc=0.818  test_acc=0.701  (2.2 min)
  epoch=011  train_acc=0.832  test_acc=0.752  (2.2 min)
  epoch=012  train_acc=0.846  test_acc=0.758  (2.2 min)
  epoch=013  train_acc=0.843  test_acc=0.730  (2.2 min)
  epoch=014  train_acc=0.849  test_acc=0.741  (2.2 min)
  epoch=015  train_acc=0.855  test_acc=0.730  (2.2 min)
  epoch=016  train_acc=0.858  test_acc=0.765  (2.2 min)
  epoch=017  train_acc=0.861  test_acc=0.800  (2.2 min)
  ep

,run_label,task_key,task_name,pair_seed,pair_role,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hidden_tau_mem_min_ms,hidden_tau_mem_max_ms,hidden_tau_mem_mean_ms,hidden_tau_mem_std_ms,linear_sync_verified,checkpoint,elapsed_s,final_test_acc,final_test_loss
0,seeded_run_2,2class,binary_parity,231,heterogeneous_sampled,lognormal,10.0,25.226301,2.072141,99.749887,37.873874,31.872785,True,C:\Users\Priya\Desktop\research project (SNN I...,3211.201697,0.762367,0.490538
1,seeded_run_2,2class,binary_parity,231,homogeneous_anchor,lognormal,10.0,25.226301,25.226301,25.226301,25.226301,0.000000,True,C:\Users\Priya\Desktop\research project (SNN I...,3230.194544,0.733657,0.615711


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,231,2class,binary_parity,lognormal,10.0,25.226301,2.072141,99.749887,37.873874,31.872785,29,1,True,6


In [16]:
# Train or reuse seed pair 241
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(MASTER_SEEDS[4])
display(seed_df)
display(pair_df[pair_df["pair_seed"] == MASTER_SEEDS[4]].reset_index(drop=True))


Seed 241 :: training homogeneous_anchor...
  epoch=001  train_acc=0.501  test_acc=0.506  (2.2 min)
  epoch=002  train_acc=0.498  test_acc=0.506  (2.2 min)
  epoch=003  train_acc=0.512  test_acc=0.521  (2.2 min)
  epoch=004  train_acc=0.626  test_acc=0.706  (2.2 min)
  epoch=005  train_acc=0.671  test_acc=0.718  (2.2 min)
  epoch=006  train_acc=0.712  test_acc=0.678  (2.2 min)
  epoch=007  train_acc=0.758  test_acc=0.780  (2.2 min)
  epoch=008  train_acc=0.790  test_acc=0.676  (2.1 min)
  epoch=009  train_acc=0.803  test_acc=0.756  (2.1 min)
  epoch=010  train_acc=0.816  test_acc=0.642  (2.1 min)
  epoch=011  train_acc=0.820  test_acc=0.758  (2.1 min)
  epoch=012  train_acc=0.830  test_acc=0.764  (2.1 min)
  epoch=013  train_acc=0.838  test_acc=0.769  (2.1 min)
  epoch=014  train_acc=0.844  test_acc=0.761  (2.1 min)
  epoch=015  train_acc=0.856  test_acc=0.726  (2.1 min)
  epoch=016  train_acc=0.857  test_acc=0.713  (2.1 min)
  epoch=017  train_acc=0.863  test_acc=0.788  (2.1 min)
  ep

,run_label,task_key,task_name,pair_seed,pair_role,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hidden_tau_mem_min_ms,hidden_tau_mem_max_ms,hidden_tau_mem_mean_ms,hidden_tau_mem_std_ms,linear_sync_verified,checkpoint,elapsed_s,final_test_acc,final_test_loss
0,seeded_run_2,2class,binary_parity,241,heterogeneous_sampled,lognormal,10.0,29.445742,3.794687,99.749887,41.174600,31.371058,True,C:\Users\Priya\Desktop\research project (SNN I...,3241.868522,0.827297,0.410522
1,seeded_run_2,2class,binary_parity,241,homogeneous_anchor,lognormal,10.0,29.445742,29.445742,29.445742,29.445742,0.000000,True,C:\Users\Priya\Desktop\research project (SNN I...,3200.220661,0.758834,0.548316


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,241,2class,binary_parity,lognormal,10.0,29.445742,3.794687,99.749887,41.1746,31.371058,29,1,True,6


In [17]:
pair_df, checkpoint_df, paired_acc_df = show_run_status()

if pair_df.empty:
    print("No saved seed summaries yet.")
else:
    display(pair_df)

if checkpoint_df.empty:
    print("No saved checkpoint summaries yet.")
else:
    display(checkpoint_df)

if not paired_acc_df.empty:
    display(paired_acc_df)
    print(f"Saved paired accuracy view to: {PAIRED_ACC_CSV}")

,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,201,2class,binary_parity,lognormal,10.0,23.203168,3.287719,99.749887,34.831960,31.291707,30,1,True,6
1,211,2class,binary_parity,lognormal,10.0,28.072028,2.590169,99.749887,37.446883,26.115192,32,1,True,6
2,221,2class,binary_parity,lognormal,10.0,24.333673,4.329914,99.749887,32.422747,22.852126,32,1,True,6
3,231,2class,binary_parity,lognormal,10.0,25.226301,2.072141,99.749887,37.873874,31.872785,29,1,True,6
4,241,2class,binary_parity,lognormal,10.0,29.445742,3.794687,99.749887,41.174600,31.371058,29,1,True,6


,run_label,task_key,task_name,pair_seed,pair_role,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hidden_tau_mem_min_ms,hidden_tau_mem_max_ms,hidden_tau_mem_mean_ms,hidden_tau_mem_std_ms,linear_sync_verified,checkpoint,elapsed_s,final_test_acc,final_test_loss
0,seeded_run_2,2class,binary_parity,201,heterogeneous_sampled,lognormal,10.0,23.203168,3.287719,99.749887,34.831960,31.291707,True,C:\Users\Priya\Desktop\research project (SNN I...,3201.039513,0.864841,0.360071
1,seeded_run_2,2class,binary_parity,201,homogeneous_anchor,lognormal,10.0,23.203168,23.203168,23.203168,23.203168,0.000000,True,C:\Users\Priya\Desktop\research project (SNN I...,3918.951533,0.844965,0.404792
2,seeded_run_2,2class,binary_parity,211,heterogeneous_sampled,lognormal,10.0,28.072028,2.590169,99.749887,37.446883,26.115192,True,C:\Users\Priya\Desktop\research project (SNN I...,3222.487701,0.506184,0.582101
3,seeded_run_2,2class,binary_parity,211,homogeneous_anchor,lognormal,10.0,28.072028,28.072028,28.072028,28.072028,0.000000,True,C:\Users\Priya\Desktop\research project (SNN I...,3238.934857,0.758834,0.504108
4,seeded_run_2,2class,binary_parity,221,heterogeneous_sampled,lognormal,10.0,24.333673,4.329914,99.749887,32.422747,22.852126,True,C:\Users\Priya\Desktop\research project (SNN I...,3211.423915,0.822880,0.446224
5,seeded_run_2,2class,binary_parity,221,homogeneous_anchor,lognormal,10.0,24.333673,24.333673,24.333673,24.333673,0.000000,True,C:\Users\Priya\Desktop\research project (SNN I...,3204.421216,0.809187,0.502720
6,seeded_run_2,2class,binary_parity,231,heterogeneous_sampled,lognormal,10.0,25.226301,2.072141,99.749887,37.873874,31.872785,True,C:\Users\Priya\Desktop\research project (SNN I...,3211.201697,0.762367,0.490538
7,seeded_run_2,2class,binary_parity,231,homogeneous_anchor,lognormal,10.0,25.226301,25.226301,25.226301,25.226301,0.000000,True,C:\Users\Priya\Desktop\research project (SNN I...,3230.194544,0.733657,0.615711
8,seeded_run_2,2class,binary_parity,241,heterogeneous_sampled,lognormal,10.0,29.445742,3.794687,99.749887,41.174600,31.371058,True,C:\Users\Priya\Desktop\research project (SNN I...,3241.868522,0.827297,0.410522
9,seeded_run_2,2class,binary_parity,241,homogeneous_anchor,lognormal,10.0,29.445742,29.445742,29.445742,29.445742,0.000000,True,C:\Users\Priya\Desktop\research project (SNN I...,3200.220661,0.758834,0.548316


pair_role,pair_seed,heterogeneous_sampled,homogeneous_anchor,hetero_minus_hom
0,201,0.864841,0.844965,0.019876
1,211,0.506184,0.758834,-0.252650
2,221,0.822880,0.809187,0.013693
3,231,0.762367,0.733657,0.028710
4,241,0.827297,0.758834,0.068463


Saved paired accuracy view to: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\Checkpoints\SeededRuns\seeded_run_2\2class_lognormal\seeded_run_2_paired_accuracy.csv
